In [11]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00


In [26]:
!pip install pyngrok

In [27]:
import cv2
import os
from ultralytics import YOLO
import zipfile
import os
from pyngrok import ngrok
import threading

In [3]:
# Folder containing videos
videos_folder = "/content/videos"

# Output folder
output_folder = "frames"

os.makedirs(output_folder, exist_ok=True)

# Loop through all video files
for video_file in os.listdir(videos_folder):

    if video_file.endswith(".mp4"):

        video_path = os.path.join(videos_folder, video_file)

        # Create folder for each video
        video_name = os.path.splitext(video_file)[0]
        save_folder = os.path.join(output_folder, video_name)

        os.makedirs(save_folder, exist_ok=True)

        # Read video
        cap = cv2.VideoCapture(video_path)

        frame_count = 0

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            # Save frame
            frame_path = os.path.join(
                save_folder,
                f"frame_{frame_count:05d}.jpg"
            )

            cv2.imwrite(frame_path, frame)

            frame_count += 1

        cap.release()

        print(f"{video_file}: {frame_count} frames extracted")

print("Finished.")

VID_20260504_144439.mp4: 604 frames extracted
VID_20260504_143244.mp4: 346 frames extracted
VID_20260504_143758.mp4: 640 frames extracted
VID_20260504_143535.mp4: 442 frames extracted
Finished.


In [6]:
import shutil

# Create zip file
shutil.make_archive("/content/frames", "zip", "/content/frames")

'/content/frames.zip'

In [9]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/frames.zip /content/drive/MyDrive/

Mounted at /content/drive


Model

In [13]:
zip_path = "/content/BowlingData.zip"
extract_path = "/content/BowlingData"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted.")


Dataset extracted.


In [14]:
print(os.listdir(extract_path))

['README.dataset.txt', 'train', 'test', 'README.roboflow.txt', 'valid', 'data.yaml']


In [16]:
model = YOLO("yolov8n.pt")
model.train(
    data="/content/BowlingData/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10
)

print("Training Finished!")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/BowlingData/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

In [32]:
model.save()

### Save trained model to Google Drive

In [33]:
import glob
import os

model_save_path = glob.glob('/content/runs/detect/train*/weights/best.pt')

if model_save_path:
    latest_model_path = model_save_path[0]
    print(f"Trained model found at: {latest_model_path}")

    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    drive_destination = '/content/drive/MyDrive/best_bowling_model.pt'
    !cp "{latest_model_path}" "{drive_destination}"
    print(f"Model copied to Google Drive at: {drive_destination}")
else:
    print("No trained model (best.pt) found in runs/detect/train*/weights/")


Trained model found at: /content/runs/detect/train/weights/best.pt
video 1/1 (frame 8/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 1 person, 4 bottles, 6 chairs, 4 vases, 16.5ms
video 1/1 (frame 9/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 4 bottles, 6 chairs, 3 vases, 13.3ms
video 1/1 (frame 10/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 4 bottles, 5 chairs, 3 vases, 20.3ms
video 1/1 (frame 11/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 5 bottles, 5 chairs, 3 vases, 16.1ms
video 1/1 (frame 12/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 4 bottles, 5 chairs, 3 vases, 26.1ms


INFO:werkzeug:127.0.0.1 - - [07/May/2026 14:54:33] "GET /status/a09751af HTTP/1.1" 200 -


video 1/1 (frame 13/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 4 bottles, 5 chairs, 4 vases, 25.7ms
video 1/1 (frame 14/468) /tmp/bv_uploads/a09751af_testtttt.mp4: 384x640 5 bottles, 5 chairs, 5 vases, 30.0ms
Model copied to Google Drive at: /content/drive/MyDrive/best_bowling_model.pt


In [18]:
results = model.predict(
    source="/content/videos/VID_20260504_143244.mp4",
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/346) /content/videos/VID_20260504_143244.mp4: 384x640 3 balls, 15 standing-pinss, 68.4ms
video 1/1 (frame 2/346) /content/videos/VID_20260504_143244.mp4: 384x640 3 balls, 15 standing-pinss, 16.9ms
video 1/1 (frame 3/346) /content/videos/VID_20260504_143244.mp4: 384x640 3 balls, 15 standing-pinss, 8.3ms
video 1/1 (frame 4/346) /content/videos/VID_20260504_143244.mp4: 384x640 3 balls, 15 standing-pinss, 12.8ms
video 1/1 (frame 5/346) /co

In [23]:
results = model.predict(
    source="/content/WhatsApp Video 2026-05-07 at 4.09.26 PM.mp4",
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1893) /content/WhatsApp Video 2026-05-07 at 4.09.26 PM.mp4: 640x384 12 standing-pinss, 9.2ms
video 1/1 (frame 2/1893) /content/WhatsApp Video 2026-05-07 at 4.09.26 PM.mp4: 640x384 12 standing-pinss, 6.2ms
video 1/1 (frame 3/1893) /content/WhatsApp Video 2026-05-07 at 4.09.26 PM.mp4: 640x384 12 standing-pinss, 6.2ms
video 1/1 (frame 4/1893) /content/WhatsApp Video 2026-05-07 at 4.09.26 PM.mp4: 640x384 13 standing-pinss, 6.1ms
video 1/1 

UI

In [31]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics", "flask", "pyngrok"], check=True)

import os, uuid, threading
from flask import Flask, request, jsonify, send_from_directory, Response
import cv2
from ultralytics import YOLO

app = Flask(__name__)
UPLOAD_DIR = "/tmp/bv_uploads";  os.makedirs(UPLOAD_DIR, exist_ok=True)
OUTPUT_DIR = "/tmp/bv_outputs";  os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_DIR  = "/tmp/bv_models";   os.makedirs(MODEL_DIR,  exist_ok=True)
jobs = {}

HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>🎳 BowlVision</title>
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
<style>
  :root{
    --bg:#0a0a0f;
    --surface:#13131a;
    --surface2:#1c1c28;
    --accent:#ff6b35;
    --accent2:#ffd23f;
    --green:#00e676;
    --text:#f0f0f5;
    --muted:#6b6b88;
    --border:#2a2a3e;
    --glow:rgba(255,107,53,.15);
  }
  *, *::before, *::after { margin:0; padding:0; box-sizing:border-box; }

  html, body {
    width: 100%;
    height: 100%;
    background: var(--bg);
    color: var(--text);
    font-family: 'DM Sans', sans-serif;
    overflow: hidden;
  }

  body::before {
    content: '';
    position: fixed;
    inset: 0;
    background:
      radial-gradient(ellipse 80% 50% at 20% 10%, rgba(255,107,53,.07) 0%, transparent 60%),
      radial-gradient(ellipse 60% 40% at 80% 90%, rgba(255,210,63,.05) 0%, transparent 60%);
    pointer-events: none;
    z-index: 0;
  }

  /* ── App Shell ── */
  #app {
    position: fixed;
    inset: 0;
    display: flex;
    flex-direction: column;
    z-index: 1;
  }

  /* ── Top Bar ── */
  #topbar {
    flex-shrink: 0;
    height: 60px;
    background: rgba(19,19,26,.95);
    border-bottom: 1px solid var(--border);
    display: flex;
    align-items: center;
    padding: 0 28px;
    gap: 16px;
    backdrop-filter: blur(8px);
  }
  .logo-icon {
    width: 38px; height: 38px;
    background: var(--accent);
    border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-size: 18px;
    box-shadow: 0 0 18px rgba(255,107,53,.4);
    flex-shrink: 0;
  }
  .logo-text h1 {
    font-family: 'Bebas Neue', sans-serif;
    font-size: 1.6rem;
    letter-spacing: .06em;
    line-height: 1;
  }
  .logo-text span {
    font-size: .65rem;
    color: var(--muted);
    letter-spacing: .2em;
    text-transform: uppercase;
  }
  #topbar-right {
    margin-left: auto;
    font-size: .7rem;
    color: var(--muted);
    letter-spacing: .06em;
  }

  /* ── Main Content ── */
  #main {
    flex: 1;
    display: grid;
    grid-template-columns: 340px 1fr;
    overflow: hidden;
  }

  /* ── Left Panel ── */
  #left {
    background: var(--surface);
    border-right: 1px solid var(--border);
    display: flex;
    flex-direction: column;
    overflow-y: auto;
    padding: 20px;
    gap: 0;
  }
  #left::-webkit-scrollbar { width: 4px; }
  #left::-webkit-scrollbar-track { background: transparent; }
  #left::-webkit-scrollbar-thumb { background: var(--border); border-radius: 2px; }

  .pt {
    font-family: 'Bebas Neue', sans-serif;
    font-size: .85rem;
    letter-spacing: .12em;
    color: var(--muted);
    text-transform: uppercase;
    margin-bottom: 12px;
    margin-top: 18px;
    display: flex;
    align-items: center;
    gap: 8px;
  }
  .pt:first-child { margin-top: 0; }
  .pt::after { content: ''; flex: 1; height: 1px; background: var(--border); }

  /* Dropzone */
  #dropzone {
    border: 2px dashed var(--border);
    border-radius: 13px;
    padding: 26px 16px;
    text-align: center;
    cursor: pointer;
    transition: all .25s;
  }
  #dropzone:hover, #dropzone.dragover { border-color: var(--accent); background: var(--glow); }
  #dropzone.has-file { border-color: var(--green); background: rgba(0,230,118,.05); border-style: solid; }
  #dropzone .di { font-size: 2.2rem; margin-bottom: 7px; display: block; }
  #dropzone h3 { font-size: .88rem; font-weight: 600; margin-bottom: 3px; }
  #dropzone p { font-size: .74rem; color: var(--muted); }
  #file-input { display: none; }

  /* Slider */
  .srow { margin-bottom: 4px; }
  .slabel { display: flex; justify-content: space-between; font-size: .78rem; font-weight: 500; margin-bottom: 7px; }
  .slabel .val { color: var(--accent); font-weight: 700; font-family: monospace; }
  input[type=range] { width: 100%; height: 4px; -webkit-appearance: none; background: var(--border); border-radius: 2px; outline: none; }
  input[type=range]::-webkit-slider-thumb { -webkit-appearance: none; width: 15px; height: 15px; border-radius: 50%; background: var(--accent); cursor: pointer; box-shadow: 0 0 7px rgba(255,107,53,.5); }

  /* Model zone */
  .mzone {
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 10px 14px;
    display: flex; align-items: center; gap: 9px;
    cursor: pointer;
    transition: border-color .2s;
    font-size: .78rem; color: var(--muted);
    background: var(--surface2);
  }
  .mzone:hover { border-color: var(--accent); color: var(--text); }
  .mzone.has-model { border-color: var(--green); color: var(--green); }
  #model-input { display: none; }

  /* Run button */
  #run-btn {
    width: 100%;
    margin-top: auto;
    margin-top: 20px;
    padding: 13px;
    background: var(--accent);
    color: #fff;
    font-family: 'Bebas Neue', sans-serif;
    font-size: 1.1rem;
    letter-spacing: .15em;
    border: none;
    border-radius: 11px;
    cursor: pointer;
    transition: all .2s;
    box-shadow: 0 4px 18px rgba(255,107,53,.3);
  }
  #run-btn:hover:not(:disabled) { transform: translateY(-1px); box-shadow: 0 6px 26px rgba(255,107,53,.45); }
  #run-btn:disabled { background: var(--muted); cursor: not-allowed; box-shadow: none; transform: none; }

  /* ── Right Panel ── */
  #right {
    display: flex;
    flex-direction: column;
    overflow: hidden;
    background: var(--bg);
  }

  /* Output header */
  #ohdr {
    flex-shrink: 0;
    padding: 14px 22px;
    border-bottom: 1px solid var(--border);
    display: flex;
    align-items: center;
    gap: 10px;
    background: var(--surface);
  }
  #ohdr .pt { margin: 0; }
  #sbadge {
    margin-left: auto;
    padding: 3px 11px;
    border-radius: 20px;
    font-size: .68rem;
    font-weight: 700;
    letter-spacing: .1em;
    text-transform: uppercase;
  }
  .s-idle { background: var(--surface2); color: var(--muted); }
  .s-processing { background: rgba(255,210,63,.15); color: var(--accent2); }
  .s-done { background: rgba(0,230,118,.15); color: var(--green); }
  .s-error { background: rgba(255,50,50,.15); color: #ff5555; }

  /* Output body */
  #obody {
    flex: 1;
    padding: 28px;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    overflow-y: auto;
    gap: 16px;
  }
  #obody::-webkit-scrollbar { width: 4px; }
  #obody::-webkit-scrollbar-thumb { background: var(--border); border-radius: 2px; }

  /* Idle state */
  .estate { text-align: center; color: var(--muted); }
  .estate .ei { font-size: 4rem; margin-bottom: 12px; opacity: .25; }
  .estate p { font-size: .84rem; line-height: 1.7; }

  /* ── Progress ── */
  #prog { width: 100%; max-width: 640px; display: none; }

  .prog-header {
    display: flex; align-items: baseline; justify-content: space-between;
    margin-bottom: 12px;
  }
  .prog-title { font-size: 1rem; font-weight: 600; color: var(--text); }
  .prog-pct { font-size: 1.8rem; font-weight: 700; font-family: monospace; color: var(--accent); line-height: 1; }

  .pmsg {
    font-size: .82rem; color: var(--muted); margin-bottom: 10px;
    display: flex; align-items: center; gap: 8px;
  }
  .spin {
    width: 13px; height: 13px;
    border: 2px solid var(--border);
    border-top-color: var(--accent);
    border-radius: 50%;
    animation: spin .8s linear infinite;
    display: inline-block; flex-shrink: 0;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  .pbg { height: 7px; background: var(--border); border-radius: 4px; overflow: hidden; width: 100%; margin-bottom: 20px; }
  .pfill {
    height: 100%;
    background: linear-gradient(90deg, var(--accent), var(--accent2));
    border-radius: 4px;
    transition: width .55s cubic-bezier(.4,0,.2,1);
    width: 0%;
  }

  /* Steps */
  .steps { display: flex; flex-direction: column; gap: 10px; width: 100%; }
  .step {
    display: flex; align-items: center; gap: 14px;
    padding: 13px 16px;
    border-radius: 11px;
    background: var(--surface2);
    border: 1px solid var(--border);
    font-size: .82rem;
    transition: all .35s;
  }
  .step.done { border-color: rgba(0,230,118,.3); background: rgba(0,230,118,.06); }
  .step.active { border-color: rgba(255,107,53,.45); background: rgba(255,107,53,.07); }
  .step.pending { opacity: .35; }

  .step-icon {
    width: 26px; height: 26px;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: .72rem; font-weight: 700; flex-shrink: 0;
  }
  .step.done .step-icon { background: rgba(0,230,118,.2); color: var(--green); font-size: .9rem; }
  .step.active .step-icon { background: rgba(255,107,53,.2); color: var(--accent); }
  .step.pending .step-icon { background: var(--border); color: var(--muted); }

  .step-body { flex: 1; }
  .step-label { color: var(--text); font-weight: 500; }
  .step-sub { font-size: .73rem; color: var(--muted); margin-top: 3px; }
  .step-time { font-size: .72rem; color: var(--muted); font-family: monospace; white-space: nowrap; }
  .step.done .step-time { color: var(--green); }

  /* ── Output video & stats ── */
  #outvid {
    width: 100%; max-width: 900px;
    border-radius: 11px; max-height: 55vh;
    display: none; background: #000;
  }

  #stats { width: 100%; max-width: 900px; display: none; }
  .sgrid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 10px; margin-bottom: 12px;
  }
  .sc {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 14px; text-align: center;
  }
  .sc .sv { font-family: 'Bebas Neue', sans-serif; font-size: 1.7rem; color: var(--accent); line-height: 1; }
  .sc .sl { font-size: .67rem; color: var(--muted); text-transform: uppercase; letter-spacing: .1em; margin-top: 4px; }

  .dlist {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 14px 16px;
  }
  .dlist h4 { font-size: .7rem; color: var(--muted); text-transform: uppercase; letter-spacing: .1em; margin-bottom: 10px; }
  .di2 {
    display: flex; justify-content: space-between; align-items: center;
    padding: 5px 0; border-bottom: 1px solid var(--border); font-size: .8rem;
  }
  .di2:last-child { border-bottom: none; }
  .dcnt { background: var(--accent); color: #fff; font-size: .68rem; font-weight: 700; padding: 2px 8px; border-radius: 9px; }

  #dlbtn {
    width: 100%; max-width: 900px;
    margin-top: 12px; padding: 12px;
    background: transparent;
    border: 1.5px solid var(--green); color: var(--green);
    font-family: 'DM Sans', sans-serif; font-size: .85rem; font-weight: 600;
    border-radius: 10px; cursor: pointer; transition: all .2s;
    display: none; letter-spacing: .04em;
  }
  #dlbtn:hover { background: rgba(0,230,118,.1); }

  #errmsg {
    color: #ff5555; font-size: .83rem;
    background: rgba(255,50,50,.08);
    border: 1px solid rgba(255,50,50,.2);
    border-radius: 10px; padding: 12px 16px;
    display: none; width: 100%; max-width: 640px; text-align: center;
  }

  /* ── Footer ── */
  #footer {
    flex-shrink: 0;
    text-align: center;
    padding: 10px 0;
    color: var(--muted);
    font-size: .68rem;
    letter-spacing: .06em;
    border-top: 1px solid var(--border);
    background: var(--surface);
  }

  @media (max-width: 700px) {
    #main { grid-template-columns: 1fr; grid-template-rows: auto 1fr; }
    #left { max-height: 50vh; }
    html, body { overflow: auto; }
    #app { position: relative; min-height: 100vh; }
  }
</style>
</head>
<body>
<div id="app">

  <!-- Top Bar -->
  <div id="topbar">
    <div class="logo-icon">🎳</div>
    <div class="logo-text">
      <h1>BowlVision</h1>
      <span>Object Detection · YOLOv8</span>
    </div>
    <div id="topbar-right">YOLOv8 · Ultralytics</div>
  </div>

  <!-- Main -->
  <div id="main">

    <!-- Left: Controls -->
    <div id="left">
      <div class="pt">📹 Input Video</div>
      <div id="dropzone" onclick="document.getElementById('file-input').click()">
        <input type="file" id="file-input" accept="video/*">
        <span class="di">🎬</span>
        <h3>Drop your video here</h3>
        <p>or click to browse · MP4, MOV, AVI</p>
      </div>

      <div class="pt">⚙️ Settings</div>
      <div class="srow">
        <div class="slabel">Confidence Threshold <span class="val" id="cv">0.25</span></div>
        <input type="range" id="cs" min="0.05" max="0.95" step="0.05" value="0.25"
          oninput="document.getElementById('cv').textContent=parseFloat(this.value).toFixed(2)">
      </div>

      <div class="pt">🧠 Custom Model</div>
      <div class="mzone" id="mzone" onclick="document.getElementById('model-input').click()">
        <input type="file" id="model-input" accept=".pt">
        <span>📂</span>
        <span id="mlabel">Upload your trained .pt weights (optional)</span>
      </div>

      <button id="run-btn" onclick="go()" disabled>▶ RUN DETECTION</button>
    </div>

    <!-- Right: Output -->
    <div id="right">
      <div id="ohdr">
        <div class="pt">🖥️ Detection Output</div>
        <span id="sbadge" class="s-idle">Idle</span>
      </div>

      <div id="obody">
        <!-- Idle state -->
        <div class="estate" id="es">
          <div class="ei">🎳</div>
          <p>Upload a video and click Run Detection<br>to see annotated results here</p>
        </div>

        <!-- Progress -->
        <div id="prog">
          <div class="prog-header">
            <div class="prog-title" id="prog-title">Processing…</div>
            <div class="prog-pct" id="pp">0%</div>
          </div>
          <div class="pmsg"><span class="spin"></span><span id="pmtxt">Starting…</span></div>
          <div class="pbg"><div class="pfill" id="pf"></div></div>

          <div class="steps">
            <div class="step pending" id="step1">
              <div class="step-icon">1</div>
              <div class="step-body">
                <div class="step-label">Load model</div>
                <div class="step-sub" id="step1-sub">Waiting…</div>
              </div>
              <div class="step-time" id="step1-time">—</div>
            </div>
            <div class="step pending" id="step2">
              <div class="step-icon">2</div>
              <div class="step-body">
                <div class="step-label">Open video</div>
                <div class="step-sub" id="step2-sub">Waiting…</div>
              </div>
              <div class="step-time" id="step2-time">—</div>
            </div>
            <div class="step pending" id="step3">
              <div class="step-icon">3</div>
              <div class="step-body">
                <div class="step-label">Run detection</div>
                <div class="step-sub" id="step3-sub">Waiting…</div>
              </div>
              <div class="step-time" id="step3-time">—</div>
            </div>
            <div class="step pending" id="step4">
              <div class="step-icon">4</div>
              <div class="step-body">
                <div class="step-label">Encode for browser</div>
                <div class="step-sub" id="step4-sub">Waiting…</div>
              </div>
              <div class="step-time" id="step4-time">—</div>
            </div>
          </div>
        </div>

        <!-- Error -->
        <div id="errmsg"></div>

        <!-- Result -->
        <video id="outvid" controls></video>
        <div id="stats">
          <div class="sgrid" id="sgrid"></div>
          <div class="dlist" id="dlist" style="display:none">
            <h4>Detected Classes</h4>
            <div id="dbody"></div>
          </div>
        </div>
        <button id="dlbtn" onclick="dl()">⬇ Download Annotated Video</button>
      </div>
    </div>
  </div>

  <!-- Footer -->
  <div id="footer">BowlVision · YOLOv8 Bowling Detection · Ultralytics</div>
</div>

<script>
  // ── FIX: store selected file in a variable so it survives DOM replacement ──
  let selectedFile = null;

  let jid=null, poll=null, outf=null, stepStart=null, stepTimes={}, lastStatus='';

  // Drag & drop
  const dz = document.getElementById('dropzone');
  dz.addEventListener('dragover', e => { e.preventDefault(); dz.classList.add('dragover'); });
  dz.addEventListener('dragleave', () => dz.classList.remove('dragover'));
  dz.addEventListener('drop', e => {
    e.preventDefault(); dz.classList.remove('dragover');
    const f = e.dataTransfer.files[0];
    if (f && f.type.startsWith('video/')) setVid(f);
  });
  document.getElementById('file-input').addEventListener('change', function() {
    if (this.files[0]) setVid(this.files[0]);
  });

  function setVid(f) {
    // ── FIX: cache the File object before innerHTML is replaced ──
    selectedFile = f;

    dz.classList.add('has-file');
    dz.innerHTML = `<span class="di">✅</span><h3>${f.name}</h3><p>${(f.size/1024/1024).toFixed(1)} MB · Click to change</p>`;

    // Re-attach a hidden input so the user can click to change the file
    const i = document.createElement('input');
    i.type='file'; i.id='file-input'; i.accept='video/*'; i.style.display='none';
    i.addEventListener('change', function() { if (this.files[0]) setVid(this.files[0]); });
    dz.appendChild(i);

    document.getElementById('run-btn').disabled = false;
  }

  document.getElementById('model-input').addEventListener('change', function() {
    if (this.files[0]) {
      document.getElementById('mlabel').textContent = '✅ ' + this.files[0].name;
      document.getElementById('mzone').classList.add('has-model');
    }
  });

  function setSt(s, t) {
    const b = document.getElementById('sbadge');
    b.className = 's-' + s;
    b.textContent = t;
  }

  function setStep(n, state, sub, time) {
    const el = document.getElementById('step' + n);
    el.className = 'step ' + state;
    const icon = el.querySelector('.step-icon');
    if (state === 'active') {
      icon.innerHTML = '<span class="spin" style="width:12px;height:12px;border-width:2px;border-top-color:var(--accent)"></span>';
    } else if (state === 'done') {
      icon.textContent = '✓';
    } else {
      icon.textContent = n;
    }
    if (sub !== undefined) document.getElementById('step' + n + '-sub').textContent = sub;
    if (time !== undefined) document.getElementById('step' + n + '-time').textContent = time;
  }

  function elapsed() {
    return ((Date.now() - stepStart) / 1000).toFixed(1) + 's';
  }

  async function go() {
    const mi = document.getElementById('model-input');
    const conf = document.getElementById('cs').value;

    // ── FIX: use selectedFile instead of reading from (potentially replaced) input ──
    if (!selectedFile) { alert('Please select a video file first.'); return; }

    const fd = new FormData();
    fd.append('video', selectedFile);
    fd.append('conf', conf);
    if (mi.files[0]) fd.append('model', mi.files[0]);

    // Reset UI
    document.getElementById('es').style.display = 'none';
    document.getElementById('errmsg').style.display = 'none';
    document.getElementById('outvid').style.display = 'none';
    document.getElementById('stats').style.display = 'none';
    document.getElementById('dlbtn').style.display = 'none';
    document.getElementById('prog').style.display = 'block';
    document.getElementById('prog-title').textContent = 'Processing…';
    document.querySelector('.pmsg').style.display = 'flex';
    document.getElementById('pf').style.width = '0%';
    document.getElementById('pp').textContent = '0%';
    document.getElementById('run-btn').disabled = true;
    setSt('processing', 'Processing');

    stepStart = Date.now();
    stepTimes = {};
    lastStatus = '';
    setStep(1, 'active', 'Loading weights…', '');
    setStep(2, 'pending', 'Waiting…', '—');
    setStep(3, 'pending', 'Waiting…', '—');
    setStep(4, 'pending', 'Waiting…', '—');

    try {
      const r = await fetch('/upload', { method: 'POST', body: fd });
      const d = await r.json();
      if (d.error) throw new Error(d.error);
      jid = d.job_id;
      startPoll();
    } catch(e) { showErr(e.message); }
  }

  function startPoll() {
    if (poll) clearInterval(poll);
    poll = setInterval(async () => {
      try {
        const r = await fetch('/status/' + jid);
        const d = await r.json();
        const pct = d.progress || 0;
        document.getElementById('pf').style.width = pct + '%';
        document.getElementById('pp').textContent = pct + '%';
        document.getElementById('pmtxt').textContent = d.message || '';

        const st = d.status;

        if (st === 'loading_model' && lastStatus !== 'loading_model') {
          setStep(1, 'active', 'Loading weights…', '');
          lastStatus = 'loading_model';
        }

        if (st === 'processing' && lastStatus === 'loading_model') {
          if (!stepTimes[1]) { stepTimes[1] = elapsed(); setStep(1, 'done', 'Model ready', stepTimes[1]); }
          setStep(2, 'active', 'Reading video metadata…', '');
          lastStatus = 'opening';
        }

        if (st === 'processing' && lastStatus === 'opening') {
          if (!stepTimes[2]) { stepTimes[2] = elapsed(); setStep(2, 'done', 'Video opened', stepTimes[2]); }
          const msg = d.message || '';
          const m = msg.match(/Frame\s+(\d+)\s*\/\s*(\d+)/i);
          setStep(3, 'active', m ? `${m[1]} of ${m[2]} frames processed` : 'Running…', '');
          lastStatus = 'detecting';
        }

        if (st === 'processing' && lastStatus === 'detecting') {
          const msg = d.message || '';
          if (msg.toLowerCase().includes('encod')) {
            if (!stepTimes[3]) { stepTimes[3] = elapsed(); setStep(3, 'done', 'All frames processed', stepTimes[3]); }
            setStep(4, 'active', 'Re-encoding with H.264…', '');
            lastStatus = 'encoding';
          } else {
            const m = msg.match(/Frame\s+(\d+)\s*\/\s*(\d+)/i);
            setStep(3, 'active', m ? `${m[1]} of ${m[2]} frames processed` : 'Running…', '');
          }
        }

        if (st === 'done') {
          clearInterval(poll);
          if (!stepTimes[3]) { setStep(3, 'done', 'All frames processed', elapsed()); }
          if (!stepTimes[4]) { setStep(4, 'done', 'Encoding complete', elapsed()); }
          document.querySelector('.pmsg').style.display = 'none';
          document.getElementById('prog-title').textContent = 'Detection complete ✓';
          setSt('done', 'Done ✓');
          showOut(d);
        } else if (st === 'error') {
          clearInterval(poll);
          showErr(d.message);
        }
      } catch(e) {}
    }, 900);
  }

  function showOut(d) {
    outf = d.output_file;
    setTimeout(() => {
      document.getElementById('prog').style.display = 'none';
      const v = document.getElementById('outvid');
      v.src = '/download/' + d.output_file;
      v.style.display = 'block';
      const sa = document.getElementById('stats');
      sa.style.display = 'block';
      const total = Object.values(d.detections || {}).reduce((a, b) => a + b, 0);
      document.getElementById('sgrid').innerHTML = `
        <div class="sc"><div class="sv">${d.total_frames || '—'}</div><div class="sl">Frames</div></div>
        <div class="sc"><div class="sv">${total}</div><div class="sl">Detections</div></div>
        <div class="sc"><div class="sv">${Object.keys(d.detections || {}).length}</div><div class="sl">Classes</div></div>`;
      const det = d.detections || {};
      if (Object.keys(det).length > 0) {
        document.getElementById('dlist').style.display = 'block';
        document.getElementById('dbody').innerHTML = Object.entries(det)
          .sort((a, b) => b[1] - a[1])
          .map(([c, n]) => `<div class="di2"><span>${c}</span><span class="dcnt">${n}</span></div>`)
          .join('');
      }
      document.getElementById('dlbtn').style.display = 'block';
    }, 400);
  }

  function showErr(msg) {
    setSt('error', 'Error');
    document.getElementById('prog').style.display = 'none';
    const e = document.getElementById('errmsg');
    e.textContent = '⚠ ' + msg;
    e.style.display = 'block';
    document.getElementById('run-btn').disabled = false;
  }

  function dl() {
    if (outf) window.open('/download/' + outf, '_blank');
  }
</script>
</body>
</html>"""


def process_video(job_id, video_path, model_path, conf):
    try:
        jobs[job_id] = {'status': 'loading_model', 'progress': 2, 'message': 'Loading model...'}
        model = YOLO(model_path)
        jobs[job_id] = {'status': 'processing', 'progress': 5, 'message': 'Opening video...'}

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        fps         = cap.get(cv2.CAP_PROP_FPS) or 25
        w           = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h           = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        raw_out = os.path.join(OUTPUT_DIR, f"{job_id}_raw.mp4")
        fourcc  = cv2.VideoWriter_fourcc(*'mp4v')
        writer  = cv2.VideoWriter(raw_out, fourcc, fps, (w, h))

        detections, idx = {}, 0
        for result in model.predict(source=video_path, conf=conf, save=False, stream=True):
            writer.write(result.plot())
            if result.boxes is not None:
                for box in result.boxes:
                    name = model.names[int(box.cls[0])]
                    detections[name] = detections.get(name, 0) + 1
            idx += 1
            jobs[job_id]['progress'] = min(5 + int(idx / total_frames * 88), 93)
            jobs[job_id]['message']  = f'Frame {idx}/{total_frames}'
        writer.release()

        jobs[job_id]['message'] = 'Encoding for browser...'
        web_out = os.path.join(OUTPUT_DIR, f"{job_id}_web.mp4")
        os.system(f'ffmpeg -y -i "{raw_out}" -vcodec libx264 -pix_fmt yuv420p "{web_out}" -loglevel error')
        final = web_out if os.path.exists(web_out) and os.path.getsize(web_out) > 0 else raw_out

        jobs[job_id] = {
            'status': 'done', 'progress': 100, 'message': 'Done!',
            'output_file': os.path.basename(final),
            'total_frames': total_frames,
            'detections': detections,
        }
    except Exception as e:
        jobs[job_id] = {'status': 'error', 'progress': 0, 'message': str(e)}


@app.route('/')
def index(): return HTML, 200, {'Content-Type': 'text/html'}

@app.route('/upload', methods=['POST'])
def upload():
    if 'video' not in request.files:
        return jsonify({'error': 'No video uploaded'}), 400
    video   = request.files['video']
    conf    = float(request.form.get('conf', 0.25))
    model_p = 'yolov8n.pt'
    if 'model' in request.files and request.files['model'].filename:
        mf = request.files['model']
        model_p = os.path.join(MODEL_DIR, mf.filename)
        mf.save(model_p)
    jid = str(uuid.uuid4())[:8]
    vpath = os.path.join(UPLOAD_DIR, f"{jid}_{video.filename}")
    video.save(vpath)
    jobs[jid] = {'status': 'queued', 'progress': 0, 'message': 'Queued...'}
    t = threading.Thread(target=process_video, args=(jid, vpath, model_p, conf), daemon=True)
    t.start()
    return jsonify({'job_id': jid})

@app.route('/status/<jid>')
def status(jid): return jsonify(jobs.get(jid, {'status': 'not_found'}))

@app.route('/download/<fname>')
def download(fname): return send_from_directory(OUTPUT_DIR, fname)


def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

import time; time.sleep(2)
from pyngrok import ngrok
ngrok.set_auth_token("2wsWYaK9fV1MHBinT6SthJzs3dU_3tkMvtzD2vPW9vxNZxBCs")
public_url = ngrok.connect(5000).public_url
print("\n" + "="*55)
print(f"  🎳  BowlVision is LIVE!")
print(f"  👉  Open this link:  {public_url}")
print("="*55)
print("  Upload any bowling video to run YOLOv8 detection.")
print("  To use your trained weights, upload the .pt file.\n")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



  🎳  BowlVision is LIVE!
  👉  Open this link:  https://882f-34-16-229-32.ngrok-free.app
  Upload any bowling video to run YOLOv8 detection.
  To use your trained weights, upload the .pt file.

